# M0 — MuseTalk spike

One question: **does this model run on the GPU Colab actually gave me, and how fast?**

Runbook and the stop conditions: `docs/M0_SPIKE.md` in the repo. Read §1 before you
start — it lists what you may and may not conclude from what you are about to measure.

**Set the runtime to a GPU first:** Runtime → Change runtime type → T4 GPU.

Run the cells in order. Each one is timed and prints what it did, so when something
breaks you know which step broke and how long you had spent — that log is worth as much
as the numbers.

Keep a note open. Every stumble gets one line with a timestamp; see §4.2.


## 1. What hardware did we actually get

Colab hands out T4 / L4 / A100 unpredictably, and every number below is meaningless
without this. Do not skip it and do not assume T4.


In [ ]:
import json, os, re, shutil, subprocess, sys, threading, time
from pathlib import Path

LOG = {}          # everything measured; printed as JSON at the end
NOTES = []        # setup problems, appended as you hit them


def note(msg):
    """Record a setup problem. Call this every time something goes wrong."""
    stamp = time.strftime('%H:%M')
    NOTES.append(f'{stamp}  {msg}')
    print(f'noted: {stamp}  {msg}')


def sh(cmd, check=True):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:], file=sys.stderr)
        if check:
            raise RuntimeError(f'failed ({result.returncode}): {cmd}')
    return result


gpu = sh('nvidia-smi --query-gpu=name,memory.total,driver_version '
         '--format=csv,noheader', check=False)
if gpu.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')

name, total_mib, driver = [f.strip() for f in gpu.stdout.strip().split(',')]
LOG['gpu'] = {'name': name, 'vram_total': total_mib, 'driver': driver}
LOG['python'] = sys.version.split()[0]
print(json.dumps(LOG, indent=2))


## 2. A measurement harness

MuseTalk's inference runs as a subprocess, so `torch.cuda.max_memory_allocated()` in this
notebook would report zero — it only sees this process's allocations. Peak VRAM is
sampled from `nvidia-smi` in a background thread instead, which measures the whole device
and is therefore slightly pessimistic: it includes anything else resident on the GPU.
Note that when you report it.


In [ ]:
def poll_vram(stop_event, samples, interval=0.25):
    query = ('nvidia-smi --query-gpu=memory.used '
             '--format=csv,noheader,nounits')
    while not stop_event.is_set():
        out = subprocess.run(query, shell=True, capture_output=True, text=True)
        if out.returncode == 0 and out.stdout.strip():
            samples.append(int(out.stdout.strip().splitlines()[0]))
        time.sleep(interval)


def run_timed(cmd, label):
    """Run a command, returning wall clock and peak device VRAM in MiB."""
    samples, stop = [], threading.Event()
    watcher = threading.Thread(target=poll_vram, args=(stop, samples), daemon=True)
    watcher.start()

    print(f'--- {label} ---')
    started = time.perf_counter()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.perf_counter() - started

    stop.set()
    watcher.join(timeout=2)

    tail = (result.stdout or '')[-3000:]
    if tail:
        print(tail)
    if result.returncode != 0:
        print((result.stderr or '')[-3000:], file=sys.stderr)

    record = {
        'seconds': round(elapsed, 2),
        'peak_vram_mib': max(samples) if samples else None,
        'exit_code': result.returncode,
    }
    print(f'{label}: {record["seconds"]}s, peak device VRAM '
          f'{record["peak_vram_mib"]} MiB, exit {record["exit_code"]}')
    return record, result


def probe_video(path):
    """Resolution, frame count, duration and fps of a rendered file, via ffprobe."""
    query = (
        'ffprobe -v error -select_streams v:0 '
        '-show_entries stream=width,height,nb_frames,avg_frame_rate,duration '
        f'-of json "{path}"'
    )
    out = subprocess.run(query, shell=True, capture_output=True, text=True)
    if out.returncode != 0:
        return {'error': out.stderr.strip()[-500:]}
    stream = json.loads(out.stdout)['streams'][0]
    num, _, den = stream.get('avg_frame_rate', '0/1').partition('/')
    fps = round(int(num) / int(den), 2) if den and int(den) else None
    return {
        'resolution': f"{stream.get('width')}x{stream.get('height')}",
        'frames': int(stream['nb_frames']) if stream.get('nb_frames') else None,
        'duration_s': round(float(stream['duration']), 2) if stream.get('duration') else None,
        'container_fps': fps,
    }

print('harness ready')


## 3. Clone and install

Timed, because "how long does it take to get running from nothing" is the setup-fragility
row in `PROCESS.md` §2.1, and because it is a real cost when the runtime is ephemeral and
you will pay it again tomorrow.

Expect dependency-resolution noise: Colab's preinstalled torch rarely matches a project's
pins. If pip resolves it, `note()` it and carry on. If it does not, that *is* the finding.


In [ ]:
WORK = Path('/content/m0')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

REPO = 'https://github.com/TMElyralab/MuseTalk.git'

if not (WORK / 'MuseTalk').exists():
    clone, _ = run_timed(f'git clone --depth 1 {REPO}', 'clone')
    LOG['clone'] = clone
else:
    print('already cloned')

os.chdir(WORK / 'MuseTalk')
sh('git log -1 --format="commit %h  %ad  %s" --date=short')
LOG['commit'] = sh('git rev-parse --short HEAD', check=False).stdout.strip()


In [ ]:
# Pin the commit you actually ran. An ephemeral runtime plus a moving upstream means
# 'I ran MuseTalk' is not a reproducible statement without this.
install, _ = run_timed('pip install -q -r requirements.txt 2>&1 | tail -20', 'pip install')
LOG['install'] = install
if install['exit_code'] != 0:
    note('requirements.txt did not install cleanly — see output above')


## 4. Weights

Several gigabytes. Timed and sized, because §3.4's "gap to production" needs to account
for cold-start cost, and because the README has to tell someone how long a clean clone
takes before it does anything.

Some mirrors have been flaky. A 403 or a truncated file on one checkpoint usually
succeeds on a retry — `note()` it either way, because "needs a retry" is a fragility
finding.


In [ ]:
script = next((p for p in ['download_weights.sh', 'scripts/download_weights.sh']
               if Path(p).exists()), None)

if script is None:
    note('no download_weights.sh found — check the README for the current weight setup')
    print('Files at repo root:')
    print('\n'.join(sorted(p.name for p in Path('.').iterdir())))
else:
    weights, _ = run_timed(f'bash {script} 2>&1 | tail -30', 'download weights')
    LOG['weights_download'] = weights
    if weights['exit_code'] != 0:
        note(f'{script} failed — rerun this cell once before concluding anything')

if Path('models').exists():
    sizes = sh('du -sh models/* 2>/dev/null | sort -h', check=False)
    total = sh('du -sh models 2>/dev/null', check=False).stdout.split()[0]
    LOG['weights_on_disk'] = total
    print(f'total on disk: {total}')


## 5. Find the current invocation

**Do not skip this.** MuseTalk's CLI has changed across releases — v1.5 introduced a
`--version` flag and moved the config format. The next cell reads the repo you just
cloned rather than trusting anything written here, because upstream is the authority and
this notebook is not.


In [ ]:
print('=== configs ===')
for cfg in sorted(Path('configs').rglob('*.yaml')) if Path('configs').exists() else []:
    print(f'  {cfg}')

print('\n=== inference scripts ===')
for s in sorted(Path('scripts').glob('*.py')) if Path('scripts').exists() else []:
    print(f'  {s}')

print('\n=== bundled sample assets ===')
for pattern in ('data/video/*', 'data/audio/*', 'assets/demo/*'):
    for asset in sorted(Path('.').glob(pattern))[:8]:
        print(f'  {asset}')

print('\n=== README: inference section ===')
readme = next((p for p in ['README.md', 'README_EN.md'] if Path(p).exists()), None)
if readme:
    text = Path(readme).read_text(encoding='utf-8', errors='replace')
    hits = [i for i, line in enumerate(text.splitlines())
            if re.search(r'inference\.py|realtime_inference', line)]
    lines = text.splitlines()
    for i in hits[:6]:
        print('\n'.join(lines[max(0, i - 4):i + 4]))
        print('  ...')


## 6. Cold run

Set `INFER_CMD` from what the cell above showed you, then run. The command below is the
commonly documented form and **may not match your clone** — if it fails on an unknown
argument, that is what happened, and it is a one-line note, not a crisis.

Use the bundled sample assets first. Getting *an answer* matters more than getting an
answer about your own reference video, and you can swap the inputs afterwards.

This run includes CUDA context creation, cuDNN autotune, and lazy weight loading. It will
be much slower than steady state. Report both numbers — §7 of the guide's trap table calls
this out specifically, because a cold first-frame figure makes a viable model look unusable.


In [ ]:
# ---- adjust from what section 5 printed ----
INFER_CMD = (
    'python -m scripts.inference '
    '--inference_config configs/inference/test.yaml '
    '--result_dir ./results/m0_cold'
)
# If your clone is v1.5+, it likely also wants:
#   --version v15 --unet_model_path models/musetalkV15/unet.pth \
#   --unet_config models/musetalkV15/musetalk.json
# -------------------------------------------

cold, cold_result = run_timed(INFER_CMD, 'inference (cold)')
LOG['inference_cold'] = cold

if cold['exit_code'] != 0:
    note('cold inference failed — fix the invocation before going further; '
         'see the stderr above and section 5')


## 7. Warm run

The same command again. This is the number that describes steady state, and the one that
belongs in §2.2's fps column.


In [ ]:
warm, _ = run_timed(INFER_CMD.replace('m0_cold', 'm0_warm'), 'inference (warm)')
LOG['inference_warm'] = warm

if cold.get('seconds') and warm.get('seconds'):
    ratio = round(cold['seconds'] / warm['seconds'], 2)
    LOG['cold_warm_ratio'] = ratio
    print(f'cold run was {ratio}x the warm run')
    if ratio > 1.5:
        note(f'cold start is {ratio}x warm — start_session() must do a warm-up pass, '
             'or first-frame latency will look far worse than it is')


## 8. What came out

Resolution and frame count, then the number that actually decides viability: **render time
divided by audio duration.** Below 1.0 is faster than real time. Above 1.0 means this
model on this GPU cannot keep up with a conversation, whatever its average fps looks like.


In [ ]:
outputs = sorted(Path('results').rglob('*.mp4')) if Path('results').exists() else []
print(f'{len(outputs)} rendered file(s)')

if not outputs:
    note('no output video — nothing has been proven yet; do not record any fps number')
else:
    newest = max(outputs, key=lambda p: p.stat().st_mtime)
    probe = probe_video(newest)
    LOG['output'] = {'path': str(newest), **probe}
    print(json.dumps(LOG['output'], indent=2))

    frames, duration = probe.get('frames'), probe.get('duration_s')
    render_s = warm.get('seconds')

    if frames and render_s:
        LOG['effective_fps_warm'] = round(frames / render_s, 2)
        print(f"\neffective fps (frames / warm render time): {LOG['effective_fps_warm']}")
        print('  NB: includes this build\'s per-run setup, so it is a floor on the')
        print('  steady-state rate a long-lived server would see, not a ceiling.')

    if duration and render_s:
        LOG['realtime_ratio'] = round(render_s / duration, 2)
        verdict = 'faster than real time' if LOG['realtime_ratio'] < 1 else 'SLOWER than real time'
        print(f"\nrender time / audio duration: {LOG['realtime_ratio']}  ({verdict})")

    from IPython.display import Video, display
    display(Video(str(newest), embed=True, width=480))


## 9. Identity preparation, measured separately

Only if your clone ships `scripts/realtime_inference.py`. It splits identity preprocessing
— face detection, parsing, latent encoding of the reference frames — from per-frame
inference, and that split is the single most architecturally significant thing you can
measure here.

It is what §1.2 of the architecture document turns on: if preprocessing is offline and
one-time, first-frame latency at conversation time can be low, and the artifact it
produces is cacheable per persona. If it has to happen per session, the serving story is
completely different. In this codebase that boundary is already drawn —
`prepare_identity` versus `push_audio` in `TalkingHeadRenderer` — so what you measure here
maps directly onto M2.

Run it twice: the second run should reuse cached preparation and be much faster. The
difference between the two runs *is* the preparation cost.


In [ ]:
if Path('scripts/realtime_inference.py').exists():
    RT_CMD = (
        'python -m scripts.realtime_inference '
        '--inference_config configs/inference/realtime.yaml '
        '--result_dir ./results/m0_rt --fps 25'
    )
    first, _ = run_timed(RT_CMD, 'realtime (prep + inference)')
    second, _ = run_timed(RT_CMD, 'realtime (prep cached)')
    LOG['realtime_first'] = first
    LOG['realtime_cached'] = second
    if first.get('seconds') and second.get('seconds'):
        LOG['identity_prep_s'] = round(first['seconds'] - second['seconds'], 2)
        print(f"\nidentity preparation cost: ~{LOG['identity_prep_s']}s")
        print('  One-time and offline. Slow here is fine and is the point.')
else:
    note('no scripts/realtime_inference.py in this clone — identity prep not measured '
         'separately')


## 10. The block to paste back

Everything measured, plus your setup notes. This goes into `PROCESS.md` §2.2 and §3.3,
attributed to the GPU named in it.

Add any notes you kept by hand to the `NOTES` list before running this — `note('...')`
appends, or edit the list directly.


In [ ]:
LOG['setup_notes'] = NOTES
LOG['spike_finished_at'] = time.strftime('%Y-%m-%d %H:%M')

print('=' * 70)
print(json.dumps(LOG, indent=2, default=str))
print('=' * 70)
print()
print('Paste the whole block above back into the session.')
print()
print('Then answer these three in your own words — they are the [HUMAN] part')
print('and PROCESS.md 2.3 is graded on them specifically:')
print('  1. Which model do you pick, and what is the ONE decisive criterion?')
print('  2. What is the strongest argument AGAINST your pick?')
print('  3. What would make you switch?')


---

## Setup log

Paste your `NOTES` here as you go, or keep it below by hand. One line each — this becomes
the setup-fragility row in the model-selection memo, and it is the difference between
"I picked MuseTalk" and a defensible choice.

```
HH:MM  what happened, and how long it cost
```

## Stop conditions

From `docs/M0_SPIKE.md` §5. Stop and replan if:

- four hours in with **no rendered frame at all** — not "nearly working"
- the weights will not download after a retry
- Colab keeps handing you a CPU runtime, or disconnects mid-run
- it runs but at under ~5fps at the smallest resolution

Do not spend three days on a CUDA install. The fallbacks are all cheap: try Ditto, drop
the resolution, or accept a lighter CPU-only model and report real numbers — which §5 of
the brief explicitly permits, as long as the memo says so.
